### ЗАДАЧА: Доска задач поддержки

У команды поддержки есть входящий поток задач от клиентов.
Нужно собрать удобную модель, которая позволит быстро посмотреть:
- какие задачи ещё открыты,
- сколько часов планируется по каждому клиенту,
- у какого клиента сейчас самая большая нагрузка,
- как меняется состояние доски после закрытия одной из задач.

На входе у тебя есть несколько строк с данными о задачах.
На выходе должен получиться компактный отчёт по текущей доске.


In [ ]:
# rows: ticket_id|client|title|estimate_hours|status
rows = [
    'TK-100|Acme|Ошибка в отчёте|3.5|new',
    'TK-101|Beta|Починить интеграцию|5|in_progress',
    'TK-102|Acme|Обновить доступы|2|new',
    'TK-103|Delta|Проверить выгрузку|1.5|closed',
]


class Ticket:
    allowed_statuses = {'new', 'in_progress', 'closed'}

    def __init__(self, ticket_id, client, title, estimate_hours, status):
        # TODO: сохранить ticket_id, client, title
        # TODO: hours хранить через внутреннее поле self._estimate_hours
        # TODO: значение estimate_hours пропустить через property/setter
        # TODO: проверить, что status входит в allowed_statuses, иначе ValueError
        self.ticket_id = ticket_id
        self.client = client
        self.title = title
        self.estimate_hours = estimate_hours

        if status not in self.allowed_statuses:
            raise ValueError("Недопустимый статус")
        self.status = status

    @property
    def estimate_hours(self):
        # TODO: вернуть текущее число часов
        return self._estimate_hours

    @estimate_hours.setter
    def estimate_hours(self, value):
        # TODO: привести value к float
        # TODO: если value <= 0 -> raise ValueError('Hours must be > 0')
        # TODO: сохранить результат в self._estimate_hours
        try:
            hours = float(value)
        except Exception:
            raise ValueError("Расчетное количество часов должно быть задано определенным числом")
        if hours <= 0:
            raise ValueError("Количество часов должно быть > 0")
        self._estimate_hours = hours

    def close(self):
        # TODO: перевести задачу в статус 'closed'
        self.status = 'closed'

    def reopen(self):
        # TODO: перевести задачу обратно в статус 'new'
        self.status = 'new'

    @classmethod
    def from_row(cls, row):
        # TODO: split по '|'
        # TODO: ожидать 5 частей: ticket_id, client, title, estimate_hours, status
        # TODO: вернуть Ticket(...)
        parts = row.split('|')
        if len(parts) != 5:
            raise ValueError("Некорректная строка для создания Ticket")
        ticket_id, client, title, estimate_hours_str, status = parts
        return cls(ticket_id, client, title, estimate_hours_str, status)

    def __repr__(self):
        # TODO: вернуть строку вида Ticket(ticket_id='...', client='...', status='...')
        return (f"Ticket(ticket_id='{self.ticket_id}', client='{self.client}', "
                f"title='{self.title}', estimate_hours={self.estimate_hours}, status='{self.status}')")


class TicketBoard:
    def __init__(self):
        self.tickets = []

    def add(self, ticket):
        # TODO: добавить объект Ticket в self.tickets
        self.tickets.append(ticket)

    def load(self, rows):
        # TODO: для каждой строки создать Ticket.from_row(row)
        # TODO: добавить тикет в доску через add(...)
        for row in rows:
            ticket = Ticket.from_row(row)
            self.add(ticket)

    def open_tickets(self):
        # TODO: вернуть список тикетов, у которых status != 'closed'
        return [t for t in self.tickets if t.status != 'closed']

    def by_client(self, client):
        # TODO: вернуть список тикетов только нужного клиента
        return [t for t in self.tickets if t.client == client]

    def total_hours_by_client(self):
        # TODO: собрать dict вида client -> total_hours
        # TODO: суммировать estimate_hours по каждому клиенту
        result = {}
        for t in self.tickets:
            result[t.client] = result.get(t.client, 0) + t.estimate_hours
        return result

    def busiest_client(self):
        # TODO: использовать total_hours_by_client()
        # TODO: вернуть tuple (client, total_hours) с максимумом
        totals = self.total_hours_by_client()
        if not totals:
            return None, 0
        max_client = max(totals, key=totals.get)
        return max_client, totals[max_client]


board = TicketBoard()

# TODO: загрузить строки в board
board.load(rows)

# TODO: вывести все тикеты
print("Все тикеты:")
for t in board.tickets:
    print(t)

# TODO: вывести только открытые тикеты
print("\nОткрытые задачи:")
for t in board.open_tickets():
    print(t)

# TODO: вывести задачи клиента 'Acme'
print("\nЗадачи клиента 'Acme':")
for t in board.by_client('Acme'):
    print(t)

# TODO: вывести total_hours_by_client()
print("\nОбщее время по клиентам:")
print(board.total_hours_by_client())

# TODO: вывести busiest_client()
client, hours = board.busiest_client()
print(f"\nСамый загруженный клиент: {client} с {hours} часами.")

# TODO: закрыть первую открытую задачу и снова вывести open_tickets()
open_tasks = board.open_tickets()
if open_tasks:
    first_task = open_tasks[0]
    first_task.close()
    print(f"\nЗадача {first_task.ticket_id} закрыта.")

for t in board.open_tickets():
    print(t)
